# M5 — Running Foundation-Sec-8B-Instruct on the SAFE threat evidence

This notebook runs the **real Foundation-Sec-8B-Instruct model** (4-bit Q4_K_M quantisation, served by Ollama on Colab's free T4 GPU) on the threat evidence that SAFE already produced. It replaces the rule-based fallback results with real LLM results.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.

Then run the cells from top to bottom (`Runtime → Run all` works too, but you'll be asked for your GitHub token and possibly to upload files).

| Step | Time (approx.) |
|---|---|
| Setup + installing Ollama | 2–4 min |
| Downloading the model (~5 GB) | 2–5 min |
| Reasoning, IoMT-TrafficData (100 flows) | 15–30 min |
| Reasoning, MedSec-25 (100 flows) | 15–30 min |

**Keep this browser tab open while it runs.** Free Colab can disconnect idle sessions.

## 1. Check that a GPU is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: go to Runtime -> Change runtime type -> T4 GPU"

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Get the project code from GitHub
The repository is public, so just press Enter when asked for a token.

In [ ]:
import os, getpass, subprocess

REPO_URL = "https://github.com/Sandra-Zen-S/AI-Cybersecurity-SOC.git"
BRANCH   = "main"

token = getpass.getpass("GitHub token (press Enter if the repo is public): ").strip()
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

REPO_DIR = "/content/project"
if not os.path.exists(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO_DIR], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr)
os.chdir(REPO_DIR)
print("Working in", os.getcwd())
assert os.path.exists("src/reasoning/foundation_sec.py"), "src/reasoning/foundation_sec.py not found - check REPO_URL / BRANCH"

GitHub token (press Enter if the repo is public): ··········
Cloning into '/content/project'...

Working in /content/project


## 3. Make sure the model gets enough time per flow
The original code gave the model 180 s per answer; a slow answer would silently fall back to the rule engine. This cell raises it (it does nothing if the repo already has the updated `foundation_sec.py`).

In [ ]:
p = "src/reasoning/foundation_sec.py"
s = open(p, encoding="utf-8").read()
old = "with urllib.request.urlopen(req, timeout=180) as resp:"
if old in s:
    s = s.replace(old, 'with urllib.request.urlopen(req, timeout=float(os.environ.get("FOUNDATION_SEC_TIMEOUT", "900"))) as resp:')
    open(p, "w", encoding="utf-8").write(s)
    print("Patched timeout -> 900 s")
else:
    print("Timeout already configurable - nothing to do")

Patched timeout -> 900 s


## 4. Threat-evidence files
M5 needs the evidence that M4 built from the SAFE detections:
- `results/reasoning/IoMT-TrafficData/threat_evidence.json`
- `results/reasoning/MedSec-25/threat_evidence.json`

If they are not in the repository, this cell asks you to upload them (get them from your teammate). The current rule-based results are backed up so they are not overwritten.

In [ ]:
import shutil, json
DATASETS = ["IoMT-TrafficData", "MedSec-25"]

for ds in DATASETS:
    d = f"results/reasoning/{ds}"
    os.makedirs(d, exist_ok=True)
    ev = f"{d}/threat_evidence.json"
    if not os.path.exists(ev):
        from google.colab import files
        print(f"\nUpload threat_evidence.json for {ds}:")
        up = files.upload()
        name = next(iter(up))
        shutil.move(name, ev)
    data = json.load(open(ev, encoding="utf-8-sig"))
    if data["evidence_metadata"]["dataset"] != ds:
        os.remove(ev)
        raise ValueError(f"That file is for {data['evidence_metadata']['dataset']}, not {ds}. Run this cell again and upload the right file.")
    n = len(data["evidence_items"])
    print(f"{ds}: {n} evidence items")

    # back up the rule-based outputs once
    bk = f"{d}/offline_backup"
    if not os.path.exists(bk):
        os.makedirs(bk)
        for f in ["foundation_sec_reasoning.json", "mitre_attack_summary.json", "threat_report.md", "pipeline_timing.json"]:
            if os.path.exists(f"{d}/{f}"):
                shutil.copy(f"{d}/{f}", bk)
        print(f"  backed up existing rule-based results -> {bk}")


Upload threat_evidence.json for IoMT-TrafficData:


Saving threat_evidence.json to threat_evidence.json
IoMT-TrafficData: 100 evidence items
  backed up existing rule-based results -> results/reasoning/IoMT-TrafficData/offline_backup

Upload threat_evidence.json for MedSec-25:


Saving threat_evidence (1).json to threat_evidence (1).json
MedSec-25: 100 evidence items
  backed up existing rule-based results -> results/reasoning/MedSec-25/offline_backup


## 5. Install and start Ollama (runs the model on the GPU)

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434"); print("Ollama is running"); break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/ollama.log").read()[-2000:])

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama is running


## 6. Download Foundation-Sec-8B-Instruct (4-bit, ~5 GB)
A model alias `foundation-sec` is created with an 8,192-token context (so the full evidence object fits) and temperature 0 (deterministic answers).

In [ ]:
GGUF = "hf.co/mradermacher/Foundation-Sec-8B-Instruct-GGUF:Q4_K_M"
!ollama pull {GGUF}
open("/content/Modelfile", "w").write(f"FROM {GGUF}\nPARAMETER num_ctx 8192\nPARAMETER temperature 0\n")
!ollama create foundation-sec -f /content/Modelfile
!ollama list



NAME                                                         ID              SIZE      MODIFIED               
foundation-sec:latest                                        f1447122a21a    4.9 GB    Less than a second ago    
hf.co/mradermacher/Foundation-Sec-8B-Instruct-GGUF:Q4_K_M    a35b6e9796bf    4.9 GB    48 seconds ago            


## 7. Quick test on 3 flows
Checks that the model answers with valid JSON and shows the time per flow. If `valid JSON` is `False` for all three, stop and look at the raw output printed below.

In [ ]:
import sys, json, time, importlib
os.environ["FOUNDATION_SEC_API_URL"] = "http://localhost:11434/v1/chat/completions"
os.environ["FOUNDATION_SEC_TIMEOUT"] = "900"
sys.path.insert(0, REPO_DIR)
import src.reasoning.foundation_sec as fs
importlib.reload(fs)

r = fs.FoundationSecReasoner("IoMT-TrafficData", backend="api", model_id="foundation-sec")
items = json.load(open("results/reasoning/IoMT-TrafficData/threat_evidence.json"))["evidence_items"][:3]
for it in items:
    t = time.time()
    a = r.analyse_item(it)
    m = a["reasoning_metadata"]
    print(f"{a['evidence_id']}: {time.time()-t:5.1f}s  valid JSON={m['llm_output_valid']}  "
          f"technique={a['mitre_attack'].get('technique_id')}  action={a['mitigation_directive']['action']}  "
          f"truth={(a.get('ground_truth') or {}).get('label')}")
    if not m["llm_output_valid"]:
        print("   raw output:", (m.get("raw_model_output") or "")[:800])

EVID-IOMT-0001: 100.0s  valid JSON=True  technique=T1595  action=RATE_LIMIT  truth=normal
EVID-IOMT-0002:  11.0s  valid JSON=True  technique=None  action=ALLOW_FLOW  truth=normal
EVID-IOMT-0003:  13.2s  valid JSON=True  technique=T1595  action=RATE_LIMIT  truth=normal


## 8. Full run — IoMT-TrafficData (100 flows)

In [ ]:
!cd {REPO_DIR} && python -m src.reasoning.foundation_sec --dataset IoMT-TrafficData --backend api --model-id foundation-sec

<frozen runpy>:130: RuntimeWarning: 'src.reasoning.foundation_sec' found in sys.modules after import of package 'src.reasoning', but prior to execution of 'src.reasoning.foundation_sec'; this may result in unpredictable behaviour

[M5] Reasoning over 100 evidence items (backend: api)...
  [001/100] EVID-IOMT-0001 SAFE=Normal  truth=normal           -> T1595     INFORMATIONAL ALLOW_FLOW
  [002/100] EVID-IOMT-0002 SAFE=Normal  truth=normal           -> None      INFORMATIONAL ALLOW_FLOW
  [003/100] EVID-IOMT-0003 SAFE=Normal  truth=normal           -> T1595     LOW           RATE_LIMIT
  [004/100] EVID-IOMT-0004 SAFE=Normal  truth=normal           -> None      INFORMATIONAL ALLOW_FLOW
  [005/100] EVID-IOMT-0005 SAFE=Anomaly truth=netscan          -> T1046     MEDIUM        RATE_LIMIT
  [006/100] EVID-IOMT-0006 SAFE=Normal  truth=normal           -> None      INFORMATIONAL ALLOW_FLOW
  [007/100] EVID-IOMT-0007 SAFE=Anomaly truth=camoverflow      -> T1046     MEDIUM        RATE_LIMIT
  [00

## 9. Full run — MedSec-25 (100 flows)

In [ ]:
!cd {REPO_DIR} && python -m src.reasoning.foundation_sec --dataset MedSec-25 --backend api --model-id foundation-sec

<frozen runpy>:130: RuntimeWarning: 'src.reasoning.foundation_sec' found in sys.modules after import of package 'src.reasoning', but prior to execution of 'src.reasoning.foundation_sec'; this may result in unpredictable behaviour

[M5] Reasoning over 100 evidence items (backend: api)...
  [001/100] EVID-MEDS-0001 SAFE=Normal  truth=Reconnaissance   -> None      INFORMATIONAL ALLOW_FLOW
  [002/100] EVID-MEDS-0002 SAFE=Normal  truth=Reconnaissance   -> None      INFORMATIONAL ALLOW_FLOW
  [003/100] EVID-MEDS-0003 SAFE=Normal  truth=Reconnaissance   -> None      INFORMATIONAL ALLOW_FLOW
  [004/100] EVID-MEDS-0004 SAFE=Normal  truth=Reconnaissance   -> None      LOW           ALLOW_FLOW
  [005/100] EVID-MEDS-0005 SAFE=Anomaly truth=Reconnaissance   -> T1046     LOW           RATE_LIMIT
  [006/100] EVID-MEDS-0006 SAFE=Normal  truth=Reconnaissance   -> None      INFORMATIONAL ALLOW_FLOW
  [007/100] EVID-MEDS-0007 SAFE=Anomaly truth=Reconnaissance   -> T1046     LOW           RATE_LIMIT
  [00

## 10. Results: Foundation-Sec vs rule-based fallback
`fallback_used_count` must be close to 0. If it is high, many flows were answered by the rule engine and the run does not count as an LLM result.

In [ ]:
rows = []
keys = ["attribution_accuracy_percent", "false_mitigation_rate_percent", "benign_flows_restricted",
        "attack_mitigation_coverage_percent", "missed_attack_rate_percent",
        "llm_valid_output_percent", "fallback_used_count", "mean_reasoning_latency_ms"]
for ds in DATASETS:
    d = f"results/reasoning/{ds}"
    llm = json.load(open(f"{d}/mitre_attack_summary.json"))
    off_p = f"{d}/offline_backup/mitre_attack_summary.json"
    off = json.load(open(off_p)) if os.path.exists(off_p) else {"evaluation": {}}
    print(f"\n=== {ds}  (backend: {llm['reasoning_backend']}) ===")
    print(f"{'metric':<38}{'rule-based':>12}{'Foundation-Sec':>16}")
    for k in keys:
        print(f"{k:<38}{str(off['evaluation'].get(k, '-')):>12}{str(llm['evaluation'].get(k, '-')):>16}")
    print("Techniques predicted:", llm["mitre_techniques_distribution"])
    print("SDN actions:", llm["sdn_mitigation_actions"])


=== IoMT-TrafficData  (backend: api) ===
metric                                  rule-based  Foundation-Sec
attribution_accuracy_percent                     -            10.0
false_mitigation_rate_percent                    -            16.0
benign_flows_restricted                          -               8
attack_mitigation_coverage_percent               -           100.0
missed_attack_rate_percent                       -             0.0
llm_valid_output_percent                         -           100.0
fallback_used_count                              -               0
mean_reasoning_latency_ms                        -       17904.446
Techniques predicted: {'T1595 - Network Service Scanning': 31, 'None - None': 37, 'T1046 - Network Service Scanning': 25, 'T1046 - Network Service Discovery': 2, 'T1041 - Exfiltration Over C2 Channel': 1, 'T1083 - File and Directory Discovery': 3, 'T1048 - Exfiltration Over Alternative Protocol': 1}
SDN actions: {'ALLOW_FLOW': 42, 'RATE_LIMIT': 58}

===

## 11. Download the results
Downloads `foundation_sec_results.zip` with both datasets' reasoning outputs (and the rule-based backups). Send this file — or the two `mitre_attack_summary.json` files — back for the paper, and give it to your teammate to commit to the repo.

In [ ]:
!cd {REPO_DIR} && zip -qr /content/foundation_sec_results.zip results/reasoning
from google.colab import files
files.download("/content/foundation_sec_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>